# Python Intermediate — Quick Reference

Essential Python concepts you need before diving into MCP development.

Topics covered: **List Comprehensions · F-Strings · Dicts · Classes · Closures · Modules**

---
## 1. List Comprehensions

A concise way to build lists from iterables — replaces verbose `for` loops.

**Syntax:** `[expression for item in iterable if condition]`

In [1]:
# Instead of this:
squares = []
for n in range(5):
    squares.append(n ** 2)
print(squares)

[0, 1, 4, 9, 16]


In [2]:
# Do this:
squares = [n ** 2 for n in range(5)]
print(squares)

[0, 1, 4, 9, 16]


In [3]:
# With a filter condition
evens = [n for n in range(10) if n % 2 == 0]
print(evens)

[0, 2, 4, 6, 8]


In [4]:
matrix = [[1, 2], [3, 4], [5, 6]]
matrix

[[1, 2], [3, 4], [5, 6]]

In [5]:
# Nested — flatten a 2D list
flat = [x for row in matrix for x in row]
print(flat)

[1, 2, 3, 4, 5, 6]


---
## 2. F-Strings

The modern way to embed variables and expressions inside strings (Python 3.6+).

**Syntax:** `f"text {variable} text"`

In [6]:
name = "Alice"
age = 30

# Basic insertion
print(f"Hello, {name}! You are {age} years old.")

# Inline expressions
print(f"In 5 years you'll be {age + 5}.")

Hello, Alice! You are 30 years old.
In 5 years you'll be 35.


In [7]:
# Number formatting
price = 9.5
print(f"Price: ${price:.2f}")

# Calling methods inline
tool_name = "weather_tool"
print(f"Registering: {tool_name.upper()}")

Price: $9.50
Registering: WEATHER_TOOL


---
## 3. Dictionaries

Key-value stores. The core data structure for passing structured data — JSON, configs, tool inputs/outputs.

**Syntax:** `{key: value}`

In [8]:
# Create and access
user = {"name": "Bob", "role": "admin", "active": True}

print(user["name"])           # direct access
print(user.get("email", ""))  # safe access with default

Bob



In [9]:
user

{'name': 'Bob', 'role': 'admin', 'active': True}

In [10]:
# Add, update, delete
user["email"] = "bob@example.com"
user.update({"role": "superadmin", "verified": True})
del user["active"]

print(user)

{'name': 'Bob', 'role': 'superadmin', 'email': 'bob@example.com', 'verified': True}


In [11]:
# Iterate key-value pairs
for key, value in user.items():
    print(f"{key}: {value}")

name: Bob
role: superadmin
email: bob@example.com
verified: True


In [12]:
# Dict comprehension
squared = {n: n ** 2 for n in range(5)}
print(squared)

# Nested dict
config = {
    "server": {"host": "localhost", "port": 8080},
    "debug": False,
}
print(config["server"]["port"])

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
8080


---
## 4. Classes

Blueprint for objects that bundle data (attributes) and behavior (methods).

**Syntax:** `class ClassName:`

In [13]:
class Tool:
    category = "utility"  # class-level attribute

    def __init__(self, name, description):
        self.name = name
        self.description = description

    def summary(self):
        return f"[{self.category}] {self.name}: {self.description}"


weather = Tool("weather", "Returns current weather data")
print(weather.summary())


[utility] weather: Returns current weather data


In [14]:
# Inheritance — extend an existing class
class AsyncTool(Tool):
    def __init__(self, name: str, description: str, timeout: int = 30):
        super().__init__(name, description)  # call parent __init__
        self.timeout = timeout

    def summary(self) -> str:
        base = super().summary()
        return f"{base} (timeout: {self.timeout}s)"


async_tool = AsyncTool("fetch_data", "Fetches remote data", timeout=60)
print(async_tool.summary())

[utility] fetch_data: Fetches remote data (timeout: 60s)


---
## 5. Closures

A **closure** is a function defined inside another function that **captures** variables
from the enclosing scope — even after the outer function has returned.

Closures are a lightweight alternative to classes when you need to carry a single piece
of mutable state around a behaviour. The `nonlocal` keyword lets the inner function
**reassign** (not just read) a variable from the enclosing scope.

In [15]:
# Inner function captures `prefix` from the outer scope

def make_greeter(prefix: str):
    def greet(name: str) -> str:
        return f"{prefix}, {name}!"   # `prefix` is captured — it lives on
    return greet                       # return the inner function itself

hello = make_greeter("Hello")
hi    = make_greeter("Hi")

print(hello("Alice"))   # Hello, Alice!
print(hi("Bob"))        # Hi, Bob!

# hello and hi each carry their own captured `prefix`

Hello, Alice!
Hi, Bob!


In [16]:
# nonlocal — let the inner function *reassign* an outer variable

def make_counter():
    count = 0                   # lives in the enclosing scope

    def increment():
        nonlocal count          # tell Python: `count` is the one above, not a new local
        count += 1
        return count

    return increment

counter = make_counter()
print(counter())    # 1
print(counter())    # 2
print(counter())    # 3

1
2
3


In [17]:
# Closures capture by reference — the inner function always sees the current binding

def run_scenarios():
    response = {}

    def callback(params):
        return response   # reads whatever response points to right now

    # Outer function rebinds response — callback sees the new value each time
    response = {"action": "cancel", "content": None}
    print(f"Scenario 1: {callback({})}")

    response = {"action": "accept", "content": {"date": "2024-12-27"}}
    print(f"Scenario 2: {callback({})}")

run_scenarios()


Scenario 1: {'action': 'cancel', 'content': None}
Scenario 2: {'action': 'accept', 'content': {'date': '2024-12-27'}}


In [18]:
# Combining both — nonlocal for the counter, reference capture for response
# This mirrors real MCP callback patterns

def run_scenarios():
    call_count = 0                      # inner function will reassign — needs nonlocal

    def elicitation_callback(params):
        nonlocal call_count             # reassigning, so nonlocal is required
        call_count += 1
        print(f"  [call #{call_count}] action={response['action']!r}")
        return response

    # Outer function rebinds response — callback sees the current value each time
    response = {"action": "cancel", "content": None}
    elicitation_callback(params={})

    response = {"action": "accept", "content": {"date": "2024-12-27"}}
    elicitation_callback(params={})

    print(f"Total callbacks fired: {call_count}")

run_scenarios()


  [call #1] action='cancel'
  [call #2] action='accept'
Total callbacks fired: 2


---
## 6. Modules

A module is any `.py` file. Split code across files and reuse it with `import`.

> The cells below show patterns — they won't run without the actual files, but the syntax is what matters.

In [19]:
# Import the whole module
import math_utils
print(math_utils.circle_area(5))

# Import specific names
from math_utils import circle_area, PI

# Import with alias
import math_utils as mu

# Using the stdlib math module as a real example:
import math
print(math.pi)
print(math.sqrt(16))

78.53975
3.141592653589793
4.0


In [20]:
# Import specific names from stdlib
from math import pi, sqrt
print(f"Area of circle r=5: {pi * 5**2:.2f}")

Area of circle r=5: 78.54


In [20]:
# The __name__ == '__main__' guard
# Prevents code from running when the file is imported as a module

# In a .py file you'd write:
# if __name__ == "__main__":
#     main()

# Inside a notebook __name__ is always '__main__', so this always runs:
if __name__ == "__main__":
    print("This runs when the script is executed directly, not when imported.")

This runs when the script is executed directly, not when imported.


### `__file__`, `os.path`, and `sys.executable`

Every MCP **stdio client** needs to spawn its server as a subprocess. To locate the
server script reliably — regardless of which directory you run from — every client
uses this pattern:

```python
import os, sys

_here = os.path.dirname(os.path.abspath(__file__))
server_script = os.path.join(_here, "server.py")
```

`__file__` is the path to the current script. `sys.executable` is the Python
interpreter that is currently running.

In [22]:
import os
import sys

# __file__ — path to the current .py file (not defined in notebooks)
# os.path.abspath  — resolve to a full, canonical path
# os.path.dirname  — strip the filename, keep the folder
# os.path.join     — combine path segments (handles OS separators)

# Simulating what __file__ contains in a real script:
fake_file = "/Users/me/project/elicitation/client_elicitation.py"

_here       = os.path.dirname(os.path.abspath(fake_file))
server_path = os.path.join(_here, "server_elicitation.py")

print(f"Script folder : {_here}")
print(f"Server script : {server_path}")

# sys.executable — the Python interpreter currently running this code
print(f"Python binary : {sys.executable}")

# In every MCP stdio client you will see exactly this:
#
#   _here = os.path.dirname(os.path.abspath(__file__))
#   server_params = StdioServerParameters(
#       command=sys.executable,
#       args=[os.path.join(_here, "server_elicitation.py")],
#   )
#
# sys.executable ensures the server uses the same venv/Python as the client.

Script folder : c:\Users\me\project\elicitation
Server script : c:\Users\me\project\elicitation\server_elicitation.py
Python binary : c:\Users\ryanb\coding_projects\mcp_masterclass_repo\python-mcp-masterclass\.venv\Scripts\python.exe
